# Liquidation Scenarios Engine Demo

This notebook demonstrates the configuration-driven liquidation scenarios engine for digital asset collateral.

## Features Demonstrated:
- Configuration-based parameter management
- Slippage calculation with market conditions
- Liquidation time estimation
- Multiple liquidation strategies
- Emergency scenario handling
- Flash loan integration
- Auction mechanisms
- Risk assessment and feasibility analysis

In [ ]:
import sys
import os
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Any

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

# Import configuration system
from core.config import load_config, LiquidationEngineConfig

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Load and Inspect Configuration

In [ ]:
# Load configuration
config = load_config()

print("=== Liquidation Scenarios Engine Configuration ===")
print(f"MCP Services:")
print(f"  Algorand Reader: {config.mcp_services.algorand_reader_url}")
print(f"  Market Data: {config.mcp_services.market_data_url}")

print(f"\nSlippage Calculation:")
print(f"  Max Slippage Cap: {config.slippage_calculation.max_slippage_cap:.1%}")
print(f"  Volume Ratio Power: {config.slippage_calculation.volume_ratio_power}")
print(f"  Base Slippage Factor: {config.slippage_calculation.base_slippage_factor}")

print(f"\nLiquidation Strategies:")
print(f"  Immediate: {config.liquidation_strategies.immediate.max_slippage_tolerance:.1%} max slippage, {config.liquidation_strategies.immediate.target_completion_time}h completion")
print(f"  Gradual: {config.liquidation_strategies.gradual.max_slippage_tolerance:.1%} max slippage, {config.liquidation_strategies.gradual.target_completion_time}h completion")
print(f"  Selective: {config.liquidation_strategies.selective.max_slippage_tolerance:.1%} max slippage, {config.liquidation_strategies.selective.target_completion_time}h completion")

print(f"\nFlash Loan Integration:")
print(f"  Enabled: {config.flash_loan_integration.enabled}")
print(f"  Max Amount: ${config.flash_loan_integration.max_flash_loan_amount:,.0f}")
print(f"  Fee Rate: {config.flash_loan_integration.flash_loan_fee_rate:.3%}")

## 2. Mock Data Classes for Testing

In [ ]:
@dataclass
class MockVolatilityMetrics:
    volatility_30d: float
    volatility_7d: float
    max_drawdown_30d: float
    value_at_risk_95: float

@dataclass
class MockLiquidityMetrics:
    liquidity_tier: str
    daily_volume_usd: float
    market_cap_usd: float
    depth_1_percent: float

@dataclass
class MockDigitalAsset:
    symbol: str
    price_usd: float
    volatility_metrics: MockVolatilityMetrics
    liquidity_metrics: MockLiquidityMetrics

@dataclass
class MockCollateralPosition:
    asset: MockDigitalAsset
    amount: float
    adjusted_value_usd: float

# Create sample assets
assets = {
    'ALGO': MockDigitalAsset(
        symbol='ALGO',
        price_usd=0.25,
        volatility_metrics=MockVolatilityMetrics(
            volatility_30d=0.45,
            volatility_7d=0.35,
            max_drawdown_30d=0.25,
            value_at_risk_95=0.15
        ),
        liquidity_metrics=MockLiquidityMetrics(
            liquidity_tier='high',
            daily_volume_usd=50_000_000,
            market_cap_usd=2_000_000_000,
            depth_1_percent=1_000_000
        )
    ),
    'USDC': MockDigitalAsset(
        symbol='USDC',
        price_usd=1.00,
        volatility_metrics=MockVolatilityMetrics(
            volatility_30d=0.02,
            volatility_7d=0.01,
            max_drawdown_30d=0.01,
            value_at_risk_95=0.005
        ),
        liquidity_metrics=MockLiquidityMetrics(
            liquidity_tier='high',
            daily_volume_usd=20_000_000,
            market_cap_usd=500_000_000,
            depth_1_percent=2_000_000
        )
    ),
    'SMALL_TOKEN': MockDigitalAsset(
        symbol='SMALL_TOKEN',
        price_usd=10.0,
        volatility_metrics=MockVolatilityMetrics(
            volatility_30d=0.80,
            volatility_7d=0.65,
            max_drawdown_30d=0.50,
            value_at_risk_95=0.30
        ),
        liquidity_metrics=MockLiquidityMetrics(
            liquidity_tier='low',
            daily_volume_usd=100_000,
            market_cap_usd=10_000_000,
            depth_1_percent=50_000
        )
    )
}

# Create sample positions
positions = [
    MockCollateralPosition(
        asset=assets['ALGO'],
        amount=40000,  # 40,000 ALGO
        adjusted_value_usd=10000  # $10,000
    ),
    MockCollateralPosition(
        asset=assets['USDC'],
        amount=15000,  # 15,000 USDC
        adjusted_value_usd=15000  # $15,000
    ),
    MockCollateralPosition(
        asset=assets['SMALL_TOKEN'],
        amount=500,  # 500 tokens
        adjusted_value_usd=5000  # $5,000
    )
]

print("Sample Portfolio:")
total_value = sum(pos.adjusted_value_usd for pos in positions)
for pos in positions:
    pct = pos.adjusted_value_usd / total_value * 100
    print(f"  {pos.asset.symbol}: ${pos.adjusted_value_usd:,.0f} ({pct:.1f}%) - {pos.asset.liquidity_metrics.liquidity_tier} liquidity")
print(f"Total Portfolio Value: ${total_value:,.0f}")

## 3. Slippage Analysis

In [ ]:
# Import slippage calculation function
sys.path.append(str(Path.cwd().parent / 'core'))
from liquidation_engine import calculate_liquidation_slippage

def analyze_slippage_by_size():
    """Analyze how slippage varies with liquidation size"""
    
    # Test different liquidation sizes
    liquidation_sizes = [10_000, 50_000, 100_000, 500_000, 1_000_000, 5_000_000]
    speed_factors = [1.0, 1.5, 2.0]  # Normal, urgent, immediate
    
    results = []
    
    for asset_name, asset in assets.items():
        for size in liquidation_sizes:
            for speed_factor in speed_factors:
                slippage = calculate_liquidation_slippage(
                    position_size_usd=size,
                    daily_volume_usd=asset.liquidity_metrics.daily_volume_usd,
                    market_depth_1pct=asset.liquidity_metrics.depth_1_percent,
                    liquidation_speed_factor=speed_factor,
                    config=config
                )
                
                results.append({
                    'asset': asset_name,
                    'size_usd': size,
                    'speed_factor': speed_factor,
                    'speed_label': {1.0: 'Normal', 1.5: 'Urgent', 2.0: 'Immediate'}[speed_factor],
                    'slippage': slippage,
                    'volume_ratio': size / asset.liquidity_metrics.daily_volume_usd
                })
    
    return pd.DataFrame(results)

slippage_df = analyze_slippage_by_size()

# Plot slippage analysis
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Slippage by Size for different assets
for asset in assets.keys():
    asset_data = slippage_df[(slippage_df['asset'] == asset) & (slippage_df['speed_factor'] == 1.0)]
    axes[0].plot(asset_data['size_usd'], asset_data['slippage'] * 100, marker='o', label=asset)

axes[0].set_xlabel('Liquidation Size (USD)')
axes[0].set_ylabel('Expected Slippage (%)')
axes[0].set_title('Slippage vs Liquidation Size (Normal Speed)')
axes[0].set_xscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Slippage by Speed Factor for ALGO
algo_data = slippage_df[slippage_df['asset'] == 'ALGO']
for speed_label in ['Normal', 'Urgent', 'Immediate']:
    speed_data = algo_data[algo_data['speed_label'] == speed_label]
    axes[1].plot(speed_data['size_usd'], speed_data['slippage'] * 100, marker='o', label=speed_label)

axes[1].set_xlabel('Liquidation Size (USD)')
axes[1].set_ylabel('Expected Slippage (%)')
axes[1].set_title('Slippage vs Speed Factor (ALGO)')
axes[1].set_xscale('log')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show some specific examples
print("\nSlippage Examples:")
examples = slippage_df[
    (slippage_df['size_usd'] == 100_000) & 
    (slippage_df['speed_factor'] == 1.0)
]
for _, row in examples.iterrows():
    print(f"  {row['asset']}: ${row['size_usd']:,.0f} liquidation → {row['slippage']:.2%} slippage")

## 4. Liquidation Time Estimation

In [ ]:
from liquidation_engine import estimate_liquidation_time

def analyze_liquidation_timing():
    """Analyze liquidation timing for different scenarios"""
    
    liquidation_sizes = [10_000, 50_000, 100_000, 500_000, 1_000_000]
    urgency_levels = ['patient', 'normal', 'urgent', 'immediate']
    
    results = []
    
    for asset_name, asset in assets.items():
        for size in liquidation_sizes:
            for urgency in urgency_levels:
                time_hours = estimate_liquidation_time(
                    position_size_usd=size,
                    daily_volume_usd=asset.liquidity_metrics.daily_volume_usd,
                    liquidity_tier=asset.liquidity_metrics.liquidity_tier,
                    urgency_level=urgency,
                    config=config
                )
                
                results.append({
                    'asset': asset_name,
                    'size_usd': size,
                    'urgency': urgency,
                    'time_hours': time_hours,
                    'volume_ratio': size / asset.liquidity_metrics.daily_volume_usd
                })
    
    return pd.DataFrame(results)

timing_df = analyze_liquidation_timing()

# Plot timing analysis
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Time by Size for different urgency levels (ALGO)
algo_timing = timing_df[timing_df['asset'] == 'ALGO']
for urgency in ['patient', 'normal', 'urgent', 'immediate']:
    urgency_data = algo_timing[algo_timing['urgency'] == urgency]
    axes[0].plot(urgency_data['size_usd'], urgency_data['time_hours'], marker='o', label=urgency.title())

axes[0].set_xlabel('Liquidation Size (USD)')
axes[0].set_ylabel('Estimated Time (Hours)')
axes[0].set_title('Liquidation Time vs Size by Urgency (ALGO)')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Time by Asset for normal urgency
normal_timing = timing_df[timing_df['urgency'] == 'normal']
for asset in assets.keys():
    asset_data = normal_timing[normal_timing['asset'] == asset]
    axes[1].plot(asset_data['size_usd'], asset_data['time_hours'], marker='o', label=asset)

axes[1].set_xlabel('Liquidation Size (USD)')
axes[1].set_ylabel('Estimated Time (Hours)')
axes[1].set_title('Liquidation Time vs Size by Asset (Normal Urgency)')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show timing examples
print("\nLiquidation Timing Examples ($100,000 liquidation):")
examples = timing_df[(timing_df['size_usd'] == 100_000)]
pivot = examples.pivot(index='asset', columns='urgency', values='time_hours')
print(pivot.round(1))

## 5. Emergency Scenario Analysis

In [ ]:
def analyze_emergency_scenarios():
    """Analyze different emergency liquidation scenarios"""
    
    scenarios = {
        'oracle_failure': config.emergency_scenarios.oracle_failure,
        'volatility_spike': config.emergency_scenarios.volatility_spike,
        'black_swan': config.emergency_scenarios.black_swan
    }
    
    analysis = []
    
    for scenario_name, scenario_config in scenarios.items():
        analysis.append({
            'scenario': scenario_name.replace('_', ' ').title(),
            'fallback_discount': getattr(scenario_config, 'fallback_discount', 0),
            'max_delay_hours': getattr(scenario_config, 'max_liquidation_delay', 0),
            'confidence_penalty': getattr(scenario_config, 'confidence_penalty', 0),
            'volatility_threshold': getattr(scenario_config, 'volatility_threshold', None),
            'slippage_multiplier': getattr(scenario_config, 'slippage_multiplier', None),
            'urgency_override': getattr(scenario_config, 'urgency_override', None),
            'emergency_slippage_tolerance': getattr(scenario_config, 'emergency_slippage_tolerance', None)
        })
    
    return pd.DataFrame(analysis)

emergency_df = analyze_emergency_scenarios()

print("=== Emergency Scenario Parameters ===")
print(emergency_df.to_string(index=False))

# Visualize emergency impact
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot discounts and penalties
scenarios = emergency_df['scenario'].tolist()
discounts = emergency_df['fallback_discount'].tolist()
penalties = emergency_df['confidence_penalty'].tolist()

axes[0].bar(scenarios, [d*100 for d in discounts], alpha=0.7, label='Fallback Discount')
axes[0].bar(scenarios, [p*100 for p in penalties], alpha=0.7, label='Confidence Penalty')
axes[0].set_ylabel('Percentage (%)')
axes[0].set_title('Emergency Scenario Impacts')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# Plot max delays
delays = emergency_df['max_delay_hours'].tolist()
axes[1].bar(scenarios, delays, alpha=0.7, color='orange')
axes[1].set_ylabel('Hours')
axes[1].set_title('Maximum Liquidation Delays')
axes[1].tick_params(axis='x', rotation=45)

# Create a summary of emergency vs normal liquidation
normal_liquidation = {
    'Normal': {'slippage': 0.05, 'time': 8, 'confidence': 0.85},
    'Oracle Failure': {'slippage': 0.05, 'time': 2, 'confidence': 0.85 * 0.7},  # 30% penalty
    'Volatility Spike': {'slippage': 0.05 * 1.5, 'time': 1, 'confidence': 0.85},  # 1.5x slippage
    'Black Swan': {'slippage': 0.25, 'time': 1, 'confidence': 0.60}  # Emergency tolerance
}

comparison_df = pd.DataFrame(normal_liquidation).T
comparison_df['slippage_pct'] = comparison_df['slippage'] * 100

# Plot confidence scores
axes[2].bar(comparison_df.index, comparison_df['confidence'], alpha=0.7, color='green')
axes[2].set_ylabel('Confidence Score')
axes[2].set_title('Liquidation Confidence by Scenario')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n=== Scenario Comparison ===")
print(comparison_df[['slippage_pct', 'time', 'confidence']].round(2))

## 6. Flash Loan Integration Analysis

In [ ]:
def analyze_flash_loan_costs():
    """Analyze flash loan costs for different liquidation sizes"""
    
    if not config.flash_loan_integration.enabled:
        print("Flash loan integration is disabled in configuration")
        return
    
    liquidation_amounts = [100_000, 500_000, 1_000_000, 2_500_000, 5_000_000]
    protocols = config.flash_loan_integration.supported_protocols
    
    results = []
    
    for amount in liquidation_amounts:
        if amount <= config.flash_loan_integration.max_flash_loan_amount:
            fee_amount = amount * config.flash_loan_integration.flash_loan_fee_rate
            gas_cost = config.execution_parameters.default_gas_cost_usd * config.flash_loan_integration.flash_loan_gas_multiplier
            total_cost = fee_amount + gas_cost
            cost_percentage = total_cost / amount
            
            results.append({
                'liquidation_amount': amount,
                'fee_amount': fee_amount,
                'gas_cost': gas_cost,
                'total_cost': total_cost,
                'cost_percentage': cost_percentage
            })
    
    flash_loan_df = pd.DataFrame(results)
    
    # Plot flash loan cost analysis
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot absolute costs
    axes[0].plot(flash_loan_df['liquidation_amount'], flash_loan_df['fee_amount'], 
                marker='o', label='Flash Loan Fee')
    axes[0].plot(flash_loan_df['liquidation_amount'], flash_loan_df['gas_cost'], 
                marker='s', label='Gas Cost')
    axes[0].plot(flash_loan_df['liquidation_amount'], flash_loan_df['total_cost'], 
                marker='^', label='Total Cost')
    
    axes[0].set_xlabel('Liquidation Amount (USD)')
    axes[0].set_ylabel('Cost (USD)')
    axes[0].set_title('Flash Loan Costs vs Liquidation Size')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot cost percentage
    axes[1].plot(flash_loan_df['liquidation_amount'], flash_loan_df['cost_percentage'] * 100, 
                marker='o', color='red')
    
    axes[1].set_xlabel('Liquidation Amount (USD)')
    axes[1].set_ylabel('Total Cost (%)')
    axes[1].set_title('Flash Loan Cost as Percentage of Liquidation')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("=== Flash Loan Cost Analysis ===")
    print(f"Fee Rate: {config.flash_loan_integration.flash_loan_fee_rate:.3%}")
    print(f"Max Amount: ${config.flash_loan_integration.max_flash_loan_amount:,.0f}")
    print(f"Gas Multiplier: {config.flash_loan_integration.flash_loan_gas_multiplier}x")
    print(f"Supported Protocols: {', '.join(protocols)}")
    
    print("\nCost Breakdown:")
    for _, row in flash_loan_df.iterrows():
        print(f"  ${row['liquidation_amount']:,.0f}: Fee ${row['fee_amount']:,.0f}, Gas ${row['gas_cost']:,.0f}, Total ${row['total_cost']:,.0f} ({row['cost_percentage']:.3%})")

analyze_flash_loan_costs()

## 7. Risk Assessment and Feasibility Analysis

In [ ]:
def create_risk_assessment_heatmap():
    """Create a risk assessment heatmap for different liquidation scenarios"""
    
    # Test different combinations
    liquidation_sizes = [50_000, 100_000, 500_000, 1_000_000, 2_000_000]
    asset_names = list(assets.keys())
    
    # Create risk assessment matrix
    risk_matrix = []
    
    for asset_name in asset_names:
        asset = assets[asset_name]
        row = []
        
        for size in liquidation_sizes:
            # Calculate volume impact
            volume_impact = size / asset.liquidity_metrics.daily_volume_usd
            
            # Determine feasibility level based on config thresholds
            thresholds = config.feasibility_assessment.volume_impact_thresholds
            
            if volume_impact <= thresholds['very_high']:
                risk_score = 1  # Very low risk
            elif volume_impact <= thresholds['high']:
                risk_score = 2  # Low risk
            elif volume_impact <= thresholds['medium']:
                risk_score = 3  # Medium risk
            elif volume_impact <= thresholds['low']:
                risk_score = 4  # High risk
            else:
                risk_score = 5  # Very high risk
            
            row.append(risk_score)
        
        risk_matrix.append(row)
    
    # Convert to DataFrame for plotting
    risk_df = pd.DataFrame(
        risk_matrix,
        index=asset_names,
        columns=[f"${size//1000}K" for size in liquidation_sizes]
    )
    
    # Create heatmap
    plt.figure(figsize=(10, 6))
    sns.heatmap(
        risk_df,
        annot=True,
        cmap='RdYlGn_r',
        vmin=1,
        vmax=5,
        cbar_kws={'label': 'Risk Level'},
        fmt='d'
    )
    
    plt.title('Liquidation Risk Assessment by Asset and Size')
    plt.xlabel('Liquidation Size')
    plt.ylabel('Asset')
    plt.tight_layout()
    plt.show()
    
    # Create feasibility summary
    print("=== Risk Level Legend ===")
    print("1 = Very Low Risk (Very High Feasibility)")
    print("2 = Low Risk (High Feasibility)")
    print("3 = Medium Risk (Medium Feasibility)")
    print("4 = High Risk (Low Feasibility)")
    print("5 = Very High Risk (Very Low Feasibility)")
    
    return risk_df

risk_assessment = create_risk_assessment_heatmap()

# Show volume impact thresholds
print("\n=== Volume Impact Thresholds ===")
thresholds = config.feasibility_assessment.volume_impact_thresholds
for level, threshold in thresholds.items():
    print(f"{level.replace('_', ' ').title()}: ≤ {threshold:.1%} of daily volume")

## 8. Auction Mechanism Analysis

In [ ]:
def simulate_dutch_auction():
    """Simulate a Dutch auction liquidation process"""
    
    if not config.auction_parameters.dutch_auction.enabled:
        print("Dutch auction is disabled in configuration")
        return
    
    auction_config = config.auction_parameters.dutch_auction
    
    # Simulate auction progression
    market_price = 100  # $100 asset
    intervals = 20  # Number of intervals to simulate
    
    auction_data = []
    
    for interval in range(intervals):
        minutes = interval * auction_config.time_interval_minutes
        discount = min(
            auction_config.starting_discount + (interval * auction_config.discount_increment),
            auction_config.max_discount
        )
        auction_price = market_price * (1 - discount)
        
        auction_data.append({
            'interval': interval,
            'minutes': minutes,
            'discount': discount,
            'auction_price': auction_price,
            'savings_pct': discount * 100
        })
    
    auction_df = pd.DataFrame(auction_data)
    
    # Plot auction progression
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot price decline
    axes[0].plot(auction_df['minutes'], auction_df['auction_price'], marker='o', color='red')
    axes[0].axhline(y=market_price, color='blue', linestyle='--', label='Market Price')
    axes[0].set_xlabel('Time (minutes)')
    axes[0].set_ylabel('Auction Price ($)')
    axes[0].set_title('Dutch Auction Price Progression')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot discount progression
    axes[1].plot(auction_df['minutes'], auction_df['savings_pct'], marker='s', color='green')
    axes[1].set_xlabel('Time (minutes)')
    axes[1].set_ylabel('Discount (%)')
    axes[1].set_title('Dutch Auction Discount Progression')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("=== Dutch Auction Configuration ===")
    print(f"Starting Discount: {auction_config.starting_discount:.1%}")
    print(f"Discount Increment: {auction_config.discount_increment:.1%}")
    print(f"Time Interval: {auction_config.time_interval_minutes} minutes")
    print(f"Maximum Discount: {auction_config.max_discount:.1%}")
    
    # Show key auction points
    print("\n=== Key Auction Points ===")
    key_points = auction_df[auction_df['interval'].isin([0, 5, 10, 15])]
    for _, row in key_points.iterrows():
        print(f"  {row['minutes']:3.0f} min: ${row['auction_price']:6.2f} ({row['savings_pct']:4.1f}% discount)")

simulate_dutch_auction()

## 9. Configuration Impact Analysis

In [ ]:
def compare_configuration_scenarios():
    """Compare different configuration scenarios"""
    
    # Test scenario: $500K ALGO liquidation
    test_size = 500_000
    test_asset = assets['ALGO']
    
    # Scenario 1: Current configuration (conservative)
    current_slippage = calculate_liquidation_slippage(
        test_size,
        test_asset.liquidity_metrics.daily_volume_usd,
        test_asset.liquidity_metrics.depth_1_percent,
        1.0,
        config
    )
    
    current_time = estimate_liquidation_time(
        test_size,
        test_asset.liquidity_metrics.daily_volume_usd,
        test_asset.liquidity_metrics.liquidity_tier,
        "normal",
        config
    )
    
    # Create modified configurations for comparison
    scenarios = {
        'Conservative (Current)': {
            'slippage': current_slippage,
            'time': current_time,
            'max_slippage_cap': config.slippage_calculation.max_slippage_cap,
            'base_factor': config.slippage_calculation.base_slippage_factor
        },
        'Aggressive': {
            'slippage': current_slippage * 1.5,  # Simulate higher slippage tolerance
            'time': current_time * 0.7,  # Faster execution
            'max_slippage_cap': 0.75,  # Higher cap
            'base_factor': 0.15  # Higher base factor
        },
        'Risk-Averse': {
            'slippage': current_slippage * 0.8,  # Lower slippage
            'time': current_time * 1.3,  # Slower execution
            'max_slippage_cap': 0.30,  # Lower cap
            'base_factor': 0.05  # Lower base factor
        }
    }
    
    # Create comparison DataFrame
    comparison_data = []
    
    for scenario_name, scenario_data in scenarios.items():
        recovery_rate = 1 - scenario_data['slippage']  # Simple recovery calculation
        recovery_amount = test_size * recovery_rate
        
        comparison_data.append({
            'scenario': scenario_name,
            'slippage_pct': scenario_data['slippage'] * 100,
            'time_hours': scenario_data['time'],
            'recovery_amount': recovery_amount,
            'recovery_pct': recovery_rate * 100,
            'max_slippage_cap': scenario_data['max_slippage_cap'] * 100,
            'base_factor': scenario_data['base_factor'] * 100
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Plot comparison
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Slippage comparison
    axes[0,0].bar(comparison_df['scenario'], comparison_df['slippage_pct'], alpha=0.7, color='red')
    axes[0,0].set_ylabel('Slippage (%)')
    axes[0,0].set_title('Expected Slippage by Scenario')
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # Time comparison
    axes[0,1].bar(comparison_df['scenario'], comparison_df['time_hours'], alpha=0.7, color='blue')
    axes[0,1].set_ylabel('Time (hours)')
    axes[0,1].set_title('Liquidation Time by Scenario')
    axes[0,1].tick_params(axis='x', rotation=45)
    
    # Recovery amount
    axes[1,0].bar(comparison_df['scenario'], comparison_df['recovery_amount'], alpha=0.7, color='green')
    axes[1,0].set_ylabel('Recovery Amount ($)')
    axes[1,0].set_title('Expected Recovery by Scenario')
    axes[1,0].tick_params(axis='x', rotation=45)
    
    # Configuration parameters
    x_pos = range(len(comparison_df))
    axes[1,1].bar([x - 0.2 for x in x_pos], comparison_df['max_slippage_cap'], 
                  width=0.4, alpha=0.7, label='Max Slippage Cap (%)')
    axes[1,1].bar([x + 0.2 for x in x_pos], comparison_df['base_factor'], 
                  width=0.4, alpha=0.7, label='Base Factor (%)')
    axes[1,1].set_ylabel('Percentage (%)')
    axes[1,1].set_title('Configuration Parameters')
    axes[1,1].set_xticks(x_pos)
    axes[1,1].set_xticklabels(comparison_df['scenario'], rotation=45)
    axes[1,1].legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"=== Configuration Scenario Comparison ({test_size:,.0f} ALGO Liquidation) ===")
    print(comparison_df[['scenario', 'slippage_pct', 'time_hours', 'recovery_pct']].round(2).to_string(index=False))

compare_configuration_scenarios()

## 10. Summary and Recommendations

In [ ]:
def generate_summary_report():
    """Generate a comprehensive summary report"""
    
    print("=" * 80)
    print("LIQUIDATION SCENARIOS ENGINE - SUMMARY REPORT")
    print("=" * 80)
    
    # Configuration summary
    print("\n📋 CONFIGURATION SUMMARY:")
    print(f"  • MCP Services: Algorand Reader ({config.mcp_services.algorand_reader_url}), Market Data ({config.mcp_services.market_data_url})")
    print(f"  • Max Slippage Cap: {config.slippage_calculation.max_slippage_cap:.1%}")
    print(f"  • Flash Loan Integration: {'Enabled' if config.flash_loan_integration.enabled else 'Disabled'}")
    print(f"  • Dutch Auction: {'Enabled' if config.auction_parameters.dutch_auction.enabled else 'Disabled'}")
    
    # Key insights
    print("\n🔍 KEY INSIGHTS:")
    print("  • Slippage increases non-linearly with liquidation size and urgency")
    print("  • High-liquidity assets (ALGO, USDC) offer better liquidation conditions")
    print("  • Emergency scenarios significantly impact liquidation parameters")
    print("  • Flash loans provide capital efficiency but add cost overhead")
    print("  • Dutch auctions can optimize recovery through price discovery")
    
    # Recommendations
    print("\n💡 RECOMMENDATIONS:")
    print("  1. Use gradual liquidation for sizes > 5% of daily volume")
    print("  2. Prioritize high-liquidity assets in waterfall liquidations")
    print("  3. Implement flash loans for liquidations > $1M to optimize capital")
    print("  4. Configure emergency scenarios based on market volatility")
    print("  5. Use Dutch auctions for non-urgent liquidations to maximize recovery")
    
    # Risk factors
    print("\n⚠️  RISK FACTORS:")
    print("  • Large liquidations (>10% daily volume) may have unpredictable slippage")
    print("  • Low-liquidity assets require significant time buffers")
    print("  • Emergency scenarios reduce confidence in estimations")
    print("  • Market conditions can rapidly change liquidation feasibility")
    
    # Configuration optimization suggestions
    print("\n🔧 CONFIGURATION OPTIMIZATION:")
    print("  • Consider lowering max_slippage_cap for conservative risk management")
    print("  • Adjust urgency_adjustments based on historical liquidation performance")
    print("  • Tune volume_thresholds based on actual market depth observations")
    print("  • Regular backtesting against historical liquidation events")
    
    # MCP integration status
    print("\n🔗 MCP INTEGRATION STATUS:")
    print(f"  • Algorand Reader Service: {config.mcp_services.algorand_reader_url}")
    print(f"  • Market Data Service: {config.mcp_services.market_data_url}")
    print("  • Real-time price feeds and blockchain data integration ready")
    print("  • Configuration supports dynamic parameter updates")
    
    print("\n" + "=" * 80)
    print("Report generated successfully. Configuration system is operational.")
    print("=" * 80)

generate_summary_report()

## Testing with MCP Services (Ports 8002 and 8003)

This notebook is configured to work with MCP services running on:
- **Algorand Reader MCP**: `http://localhost:8002`
- **Market Data MCP**: `http://localhost:8003`

The configuration system allows for easy switching between different service endpoints and can be updated via:
1. YAML configuration files
2. Environment variables
3. Runtime configuration updates

All liquidation scenarios and parameters are now configuration-driven, making the system highly adaptable to different market conditions and risk preferences.